In [ ]:
import PySpin
import cv2
import tkinter as tk
from tkinter import ttk, filedialog
import threading

class CameraApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Camera Acquisition App")

        # Variables to store camera settings
        self.fps = tk.IntVar(value=60)
        self.exposure = tk.IntVar(value=5000)
        self.output_filename = tk.StringVar(value='output')

        # Create GUI elements
        self.create_widgets()

        # VideoWriter and CSV file variables
        self.out = None
        self.csv_file = None
        self.stop_acquisition = threading.Event()  # Event to signal stop of acquisition

    def create_widgets(self):
        # Frame for camera settings
        settings_frame = ttk.LabelFrame(self.root, text="Camera Settings")
        settings_frame.grid(row=0, column=0, padx=10, pady=10, sticky=tk.W)

        # FPS Entry
        ttk.Label(settings_frame, text="FPS:").grid(row=0, column=0, sticky=tk.W)
        fps_entry = ttk.Entry(settings_frame, textvariable=self.fps)
        fps_entry.grid(row=0, column=1, padx=5, pady=5)

        # Exposure Entry
        ttk.Label(settings_frame, text="Exposure (us):").grid(row=1, column=0, sticky=tk.W)
        exposure_entry = ttk.Entry(settings_frame, textvariable=self.exposure)
        exposure_entry.grid(row=1, column=1, padx=5, pady=5)

        # Output Filename Entry
        ttk.Label(settings_frame, text="Output Filename:").grid(row=2, column=0, sticky=tk.W)
        output_entry = ttk.Entry(settings_frame, textvariable=self.output_filename)
        output_entry.grid(row=2, column=1, padx=5, pady=5)

        # Start Button
        start_button = ttk.Button(self.root, text="Start Acquisition", command=self.start_acquisition)
        start_button.grid(row=1, column=0, padx=10, pady=10)

        # Stop Button
        stop_button = ttk.Button(self.root, text="Stop Acquisition", command=self.stop_acquisition_process)
        stop_button.grid(row=1, column=1, padx=10, pady=10)

    def browse_output_file(self):
        filename = filedialog.asksaveasfilename(defaultextension=".avi", filetypes=[("AVI files", "*.avi")])
        if filename:
            self.output_filename.set(filename)

    def configure_camera(self, cam):
        nodemap = cam.GetNodeMap()
        
        # Set acquisition mode to continuous
        node_acquisition_mode = PySpin.CEnumerationPtr(nodemap.GetNode('AcquisitionMode'))
        node_acquisition_mode_continuous = node_acquisition_mode.GetEntryByName('Continuous')
        node_acquisition_mode.SetIntValue(node_acquisition_mode_continuous.GetValue())
        
        # Set pixel format to Mono8 (8-bit grayscale)
        node_pixel_format = PySpin.CEnumerationPtr(nodemap.GetNode('PixelFormat'))
        node_pixel_format_mono8 = node_pixel_format.GetEntryByName('Mono8')
        node_pixel_format.SetIntValue(node_pixel_format_mono8.GetValue())
        
        # Set maximum width
        node_width = PySpin.CIntegerPtr(nodemap.GetNode('Width'))
        max_width = node_width.GetMax()
        node_width.SetValue(max_width)
        
        # Set maximum height
        node_height = PySpin.CIntegerPtr(nodemap.GetNode('Height'))
        max_height = node_height.GetMax()
        node_height.SetValue(max_height)
        
        # Enable frame rate control
        node_acquisition_frame_rate_enable = PySpin.CBooleanPtr(nodemap.GetNode('AcquisitionFrameRateEnable'))
        node_acquisition_frame_rate_enable.SetValue(True)

        # Set frame rate
        node_frame_rate = PySpin.CFloatPtr(nodemap.GetNode('AcquisitionFrameRate'))
        node_frame_rate.SetValue(self.fps.get())
        
        # Turn off auto exposure
        node_exposure_auto = PySpin.CEnumerationPtr(nodemap.GetNode('ExposureAuto'))
        node_exposure_auto_off = node_exposure_auto.GetEntryByName('Off')
        node_exposure_auto.SetIntValue(node_exposure_auto_off.GetValue())
        
        # Turn off auto gain
        node_gain_auto = PySpin.CEnumerationPtr(nodemap.GetNode('GainAuto'))
        node_gain_auto_off = node_gain_auto.GetEntryByName('Off')
        node_gain_auto.SetIntValue(node_gain_auto_off.GetValue())
        
        # Set exposure time (in microseconds)
        node_exposure_time = PySpin.CFloatPtr(nodemap.GetNode('ExposureTime'))
        node_exposure_time.SetValue(self.exposure.get())
        
        # Set gain (if needed)
        node_gain = PySpin.CFloatPtr(nodemap.GetNode('Gain'))
        node_gain.SetValue(10.0)  # Example: set gain to 10 dB

    def acquire_video(self, cam):
        # Create a VideoWriter object
        fourcc = cv2.VideoWriter_fourcc(*'MJPG')
        frame_width = int(cam.Width())
        frame_height = int(cam.Height())
        self.out = cv2.VideoWriter(f'{self.output_filename.get()}.avi', fourcc, self.fps.get(), (frame_width, frame_height), isColor=False)

        try:
            # Start the acquisition
            cam.BeginAcquisition()

            while not self.stop_acquisition.is_set():
                # Retrieve next image
                image_result = cam.GetNextImage()

                if image_result.IsIncomplete():
                    print('Image incomplete: ', image_result.GetImageStatus())
                else:
                    # Convert image to numpy array (Grayscale)
                    frame = image_result.GetNDArray()

                    # Display the original image
                    resized_frame = cv2.resize(frame, (0, 0), None, 0.5, 0.5)
                    cv2.imshow('Video', resized_frame)

                    # Write frame to video file
                    self.out.write(frame)

                # Release image
                image_result.Release()

                # Check for 'Esc' key press to exit (optional)
                if cv2.waitKey(1) == 27:  # Esc key
                    break

        finally:
            # End acquisition
            cam.EndAcquisition()

            # Release VideoWriter and destroy OpenCV window
            self.out.release()
            cv2.destroyAllWindows()

    def start_acquisition(self):
        # Start acquisition in a separate thread
        self.stop_acquisition.clear()  # Clear the stop event flag
        acquisition_thread = threading.Thread(target=self.run_acquisition)
        acquisition_thread.start()

    def run_acquisition(self):
        system = PySpin.System.GetInstance()
        cam_list = system.GetCameras()

        try:
            if cam_list.GetSize() == 0:
                raise ValueError("No cameras found!")

            # Assume one camera for simplicity
            cam = cam_list.GetByIndex(0)

            # Initialize camera
            cam.Init()

            # Configure camera settings
            self.configure_camera(cam)

            # Acquire video until stop event is set
            self.acquire_video(cam)

            # Deinitialize camera
            cam.DeInit()

        except PySpin.SpinnakerException as ex:
            print('Error: %s' % ex)
            exit_code = 1

        finally:
            # Release camera and system resources
            del cam
            cam_list.Clear()
            system.ReleaseInstance()

    def stop_acquisition_process(self):
        # Set the stop event flag to stop acquisition
        self.stop_acquisition.set()

if __name__ == '__main__':
    root = tk.Tk()
    app = CameraApp(root)
    root.mainloop()